In [20]:
import json
import numpy as np
from scipy import stats
from scipy.signal import correlate
import matplotlib.pyplot as plt

# Load the multi-seed results to get the generated signals
# We need to regenerate outputs since we only saved MAE, not the signals themselves
# First let's set up and define the statistics

print("Surrogate Statistics Analysis")
print("=" * 60)

Surrogate Statistics Analysis


In [21]:
# Define theoretical statistics for each signal type

A = 1.0
omega = 2 * np.pi
dt = 0.01

def theoretical_stats(signal_type, A, omega, length, dt=0.01):
    """Compute theoretical statistics from known signal parameters."""
    
    if signal_type == 'sinusoid':
        mean = 0.0
        variance = A**2 / 2  # 0.5
        rms = A / np.sqrt(2)  # 0.707
        # Zero crossings per period = 2, periods in signal = length * dt * (omega / 2pi)
        num_periods = length * dt * (omega / (2 * np.pi))
        zero_crossings = 2 * num_periods
        # Dominant frequency
        dominant_freq = omega / (2 * np.pi)  # 1.0 Hz
        
    elif signal_type == 'triangle':
        mean = 0.0
        variance = A**2 / 3  # 0.333
        rms = A / np.sqrt(3)  # 0.577
        num_periods = length * dt * (omega / (2 * np.pi))
        zero_crossings = 2 * num_periods
        dominant_freq = omega / (2 * np.pi)
        
    elif signal_type == 'square':
        mean = 0.0
        variance = A**2  # 1.0
        rms = A  # 1.0
        num_periods = length * dt * (omega / (2 * np.pi))
        zero_crossings = 2 * num_periods
        dominant_freq = omega / (2 * np.pi)
        
    elif signal_type == 'delta':
        mean = A / length  # one spike of height A in length samples
        variance = (A**2) / length - (A / length)**2  # approximately A²/length
        rms = np.sqrt(A**2 / length)
        zero_crossings = 2  # signal goes 0 → A → 0
        dominant_freq = None  # flat spectrum
    
    return {
        'mean': mean,
        'variance': variance,
        'rms': rms,
        'zero_crossings': zero_crossings,
        'dominant_freq': dominant_freq,
    }

# Print theoretical values
for sig_type in ['sinusoid', 'triangle', 'square', 'delta']:
    th = theoretical_stats(sig_type, A, omega, 200)
    print(f"\n{sig_type.upper()} theoretical statistics:")
    for k, v in th.items():
        print(f"  {k}: {v:.4f}" if v is not None else f"  {k}: None (flat spectrum)")


SINUSOID theoretical statistics:
  mean: 0.0000
  variance: 0.5000
  rms: 0.7071
  zero_crossings: 4.0000
  dominant_freq: 1.0000

TRIANGLE theoretical statistics:
  mean: 0.0000
  variance: 0.3333
  rms: 0.5774
  zero_crossings: 4.0000
  dominant_freq: 1.0000

SQUARE theoretical statistics:
  mean: 0.0000
  variance: 1.0000
  rms: 1.0000
  zero_crossings: 4.0000
  dominant_freq: 1.0000

DELTA theoretical statistics:
  mean: 0.0050
  variance: 0.0050
  rms: 0.0707
  zero_crossings: 2.0000
  dominant_freq: None (flat spectrum)


In [22]:
# Functions to compute empirical statistics from a signal

def compute_empirical_stats(signal):
    """Compute surrogate statistics from a generated signal."""
    mean = np.mean(signal)
    variance = np.var(signal)
    rms = np.sqrt(np.mean(signal**2))
    
    # Zero crossings
    zero_crossings = np.sum(np.diff(np.sign(signal)) != 0)
    
    # Dominant frequency via FFT
    N = len(signal)
    freqs = np.fft.fftfreq(N, d=0.01)
    spectrum = np.abs(np.fft.fft(signal))
    pos_mask = freqs > 0
    pos_freqs = freqs[pos_mask]
    pos_spectrum = spectrum[pos_mask]
    dominant_freq = pos_freqs[np.argmax(pos_spectrum)]
    
    # Energy at dominant frequency vs total energy
    total_energy = np.sum(pos_spectrum**2)
    peak_idx = np.argmax(pos_spectrum)
    # Take energy in a small window around peak
    window = max(1, N // 100)
    peak_energy = np.sum(pos_spectrum[max(0,peak_idx-window):peak_idx+window+1]**2)
    energy_ratio = peak_energy / total_energy if total_energy > 0 else 0
    
    return {
        'mean': mean,
        'variance': variance,
        'rms': rms,
        'zero_crossings': zero_crossings,
        'dominant_freq': dominant_freq,
        'energy_ratio': energy_ratio,
    }

print("Empirical statistics functions defined.")

Empirical statistics functions defined.


In [23]:
# Now we need to regenerate signals from the trained models
# We'll load each model and generate outputs

import os
import torch
import torch.nn as nn
import torch.nn.functional as F

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

block_size = 512
vocab_size = 10
n_head = 4
n_layer = 4
n_embd = 128
dropout = 0.0

def discretize(y_norm, precision=3, base=10):
    tokens = []
    for val in y_norm:
        val = np.clip(val, 0, 1 - 1e-9)
        digits = []
        remaining = val
        for _ in range(precision):
            remaining *= base
            digit = int(remaining)
            digits.append(digit)
            remaining -= digit
        tokens.extend(digits)
    return tokens

def undiscretize(tokens, precision=3, base=10):
    values = []
    for i in range(0, len(tokens), precision):
        chunk = tokens[i:i+precision]
        if len(chunk) < precision:
            break
        val = 0
        for j, d in enumerate(chunk):
            val += d / (base ** (j + 1))
        values.append(val)
    return np.array(values)

def generate_signal(signal_type, A, omega, length, phase=0.0, dt=0.01):
    t = np.arange(length) * dt
    if signal_type == 'sinusoid':
        return A * np.sin(omega * t + phase)
    elif signal_type == 'triangle':
        return A * (2 / np.pi) * np.arcsin(np.sin(omega * t + phase))
    elif signal_type == 'square':
        return A * np.sign(np.sin(omega * t + phase))
    elif signal_type == 'delta':
        y = np.zeros(length)
        y[length // 2] = A
        return y

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * C ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.GELU(),
            nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout))
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class SignalTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

print("Model and functions defined.")

Model and functions defined.


In [24]:
# Generate outputs from saved models and compute surrogate statistics

signal_types = ['sinusoid', 'triangle', 'square', 'delta']
noise_levels = [0.0, 0.1, 0.2, 0.5, 1.0]
num_generations = 10

A = 1.0
omega = 2 * np.pi

all_surrogate_results = {}

for sig_type in signal_types:
    all_surrogate_results[sig_type] = {}
    
    for sigma in noise_levels:
        # Build model path
        if sig_type == 'sinusoid' and sigma == 0.0:
            model_path = os.path.expanduser('~/research-project/models/sinusoid/sinusoid_level1.pt')
        elif sig_type == 'sinusoid':
            model_path = os.path.expanduser(f'~/research-project/models/sinusoid/level2_sigma{sigma}.pt')
        else:
            model_path = os.path.expanduser(f'~/research-project/models/sinusoid/{sig_type}_sigma{sigma}.pt')
        
        if not os.path.exists(model_path):
            print(f"  {sig_type} σ={sigma}: model not found at {model_path}, skipping")
            continue
        
        # FIXED loading
        model = SignalTransformer().to(device)
        checkpoint = torch.load(model_path, map_location=device)
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
        else:
            model.load_state_dict(checkpoint)
        model.eval()
        
        y_range = A + 3 * sigma if sigma > 0 else A
        
        empirical_stats_list = []
        
        for gen_idx in range(num_generations):
            # Create context
            if sig_type == 'delta':
                y_ctx = np.zeros(50)
                y_ctx[25] = A
            else:
                phase = gen_idx * 0.1
                y_ctx = generate_signal(sig_type, A, omega, 50, phase=phase)
            
            y_ctx_norm = (y_ctx - (-y_range)) / (2 * y_range)
            y_ctx_norm = np.clip(y_ctx_norm, 0, 1 - 1e-9)
            ctx_tokens = discretize(y_ctx_norm)
            ctx_tensor = torch.tensor(ctx_tokens, dtype=torch.long).unsqueeze(0).to(device)
            
            with torch.no_grad():
                gen = model.generate(ctx_tensor, 600)
            
            gen_tokens = gen.tolist()[0]
            gen_values_norm = undiscretize(gen_tokens)
            gen_values = gen_values_norm * (2 * y_range) + (-y_range)
            generated = gen_values[50:]
            
            emp_stats = compute_empirical_stats(generated)
            empirical_stats_list.append(emp_stats)
        
        all_surrogate_results[sig_type][sigma] = empirical_stats_list
        
        means = [s['mean'] for s in empirical_stats_list]
        variances = [s['variance'] for s in empirical_stats_list]
        print(f"{sig_type} σ={sigma}: mean={np.mean(means):.4f}±{np.std(means):.4f}, var={np.mean(variances):.4f}±{np.std(variances):.4f}")

print("\nAll surrogate statistics computed!")

sinusoid σ=0.0: mean=-0.0003±0.0016, var=0.5004±0.0015
sinusoid σ=0.1: mean=-0.0023±0.0137, var=0.5226±0.0149
sinusoid σ=0.2: mean=0.0030±0.0242, var=0.5571±0.0256
sinusoid σ=0.5: mean=0.0212±0.0300, var=0.7907±0.0734
sinusoid σ=1.0: mean=-0.0355±0.1070, var=1.4977±0.1929
triangle σ=0.0: mean=-0.0010±0.0000, var=0.3333±0.0001
triangle σ=0.1: mean=0.0070±0.0125, var=0.3589±0.0399
triangle σ=0.2: mean=-0.0152±0.0374, var=0.4015±0.1075
triangle σ=0.5: mean=0.0137±0.0291, var=0.5801±0.0459
triangle σ=1.0: mean=-0.0389±0.0880, var=1.3769±0.1385
square σ=0.0: mean=-0.0019±0.0027, var=0.9974±0.0016
square σ=0.1: mean=-0.0694±0.0691, var=0.8100±0.1641
square σ=0.2: mean=-0.0540±0.0511, var=0.9766±0.0703
square σ=0.5: mean=-0.0068±0.0434, var=1.4188±0.2047
square σ=1.0: mean=-0.0679±0.1698, var=4.6112±0.9143
delta σ=0.0: mean=0.0050±0.0000, var=0.0050±0.0000
delta σ=0.1: mean=-0.0242±0.0106, var=0.0333±0.0301
delta σ=0.2: mean=-0.0148±0.0125, var=0.0499±0.0060
delta σ=0.5: mean=-0.0557±0.0442, 

In [25]:
# Load multi-seed results and use the 10 independently trained models
# Each seed = one independent model = one statistical unit

import json

with open(os.path.expanduser('~/research-project/results/multi_seed/all_results.json'), 'r') as f:
    multi_seed_data = json.load(f)

signal_types = ['sinusoid', 'triangle', 'square', 'delta']
noise_levels = [0.0, 0.1, 0.2, 0.5, 1.0]

# We need to regenerate outputs from each of the 10 seed models
# But we don't have individual model files for each seed from multi_seed run
# We DO have individual model files from the single-run experiments

# Check what models we have
import glob
model_dir = os.path.expanduser('~/research-project/models/sinusoid/')
print("Available models:")
for f in sorted(glob.glob(model_dir + '*.pt')):
    print(f"  {os.path.basename(f)}")

Available models:
  delta_sigma0.0.pt
  delta_sigma0.1.pt
  delta_sigma0.2.pt
  delta_sigma0.5.pt
  delta_sigma1.0.pt
  level2_sigma0.0.pt
  level2_sigma0.1.pt
  level2_sigma0.2.pt
  level2_sigma0.5.pt
  level2_sigma1.0.pt
  sinusoid_level1.pt
  square_sigma0.0.pt
  square_sigma0.1.pt
  square_sigma0.2.pt
  square_sigma0.5.pt
  square_sigma1.0.pt
  triangle_sigma0.0.pt
  triangle_sigma0.1.pt
  triangle_sigma0.2.pt
  triangle_sigma0.5.pt
  triangle_sigma1.0.pt


In [26]:
# Formal hypothesis testing
# NOTE: These are 10 rollouts from 1 model per condition (exploratory).
# For publication-quality tests, we would use the 10 independently trained seed models.

print("SURROGATE STATISTICS: Hypothesis Testing (Exploratory)")
print("NOTE: 10 rollouts from 1 model per condition. Tests are exploratory.")
print("=" * 90)

stat_names = ['mean', 'variance', 'dominant_freq', 'energy_ratio']

for sig_type in ['sinusoid', 'triangle', 'square']:
    theoretical = theoretical_stats(sig_type, A, omega, 200)
    
    # Compute reference statistics from exact clean signal (not formulas)
    if sig_type == 'delta':
        y_ref = np.zeros(200)
    else:
        y_ref = generate_signal(sig_type, A, omega, 200, phase=0.0)
    ref_stats = compute_empirical_stats(y_ref)
    
    print(f"\n{'='*70}")
    print(f"{sig_type.upper()}")
    print(f"{'='*70}")
    print(f"{'σ':<6} {'Statistic':<18} {'Reference':<14} {'Empirical':<22} {'p-value':<10} {'Sig?'}")
    print("-" * 82)
    
    for sigma in noise_levels:
        if sigma not in all_surrogate_results[sig_type]:
            continue
        
        emp_list = all_surrogate_results[sig_type][sigma]
        
        for stat_name in stat_names:
            ref_val = ref_stats[stat_name]
            emp_vals = [s[stat_name] for s in emp_list]
            
            if ref_val is None or np.std(emp_vals) == 0:
                continue
            
            t_stat, p_val = stats.ttest_1samp(emp_vals, ref_val)
            sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
            
            print(f"{sigma:<6} {stat_name:<18} {ref_val:<14.4f} {np.mean(emp_vals):.4f}±{np.std(emp_vals):.4f}   {p_val:<10.4f} {sig}")
        
        print()

print("\n*** p<0.001  ** p<0.01  * p<0.05  ns = not significant (exploratory)")

SURROGATE STATISTICS: Hypothesis Testing (Exploratory)
NOTE: 10 rollouts from 1 model per condition. Tests are exploratory.

SINUSOID
σ      Statistic          Reference      Empirical              p-value    Sig?
----------------------------------------------------------------------------------
0.0    mean               0.0000         -0.0003±0.0016   0.5811     ns
0.0    variance           0.5000         0.5004±0.0015   0.3970     ns
0.0    energy_ratio       1.0000         1.0000±0.0000   0.2055     ns

0.1    mean               0.0000         -0.0023±0.0137   0.6284     ns
0.1    variance           0.5000         0.5226±0.0149   0.0014     **
0.1    energy_ratio       1.0000         0.9320±0.1423   0.1854     ns

0.2    mean               0.0000         0.0030±0.0242   0.7140     ns
0.2    variance           0.5000         0.5571±0.0256   0.0001     ***
0.2    energy_ratio       1.0000         0.8970±0.0406   0.0000     ***

0.5    mean               0.0000         0.0212±0.0300   

In [27]:
# Hotelling T² test using non-redundant statistics: mean, variance, dominant_freq
# Dropped RMS (redundant with variance) and zero_crossings (poorly defined for delta)

from scipy.stats import f as f_dist

print("HOTELLING T² TEST (Exploratory)")
print("Statistics: mean, variance, dominant frequency")
print("=" * 70)

for sig_type in ['sinusoid', 'triangle', 'square']:
    # Reference from exact clean signal
    y_ref = generate_signal(sig_type, A, omega, 200, phase=0.0)
    ref_stats = compute_empirical_stats(y_ref)
    
    mu_0 = np.array([ref_stats['mean'], ref_stats['variance'], ref_stats['dominant_freq']])
    
    print(f"\n--- {sig_type.upper()} ---")
    print(f"Reference: mean={mu_0[0]:.4f}, var={mu_0[1]:.4f}, freq={mu_0[2]:.4f}")
    print(f"{'σ':<8} {'T²':<12} {'F-stat':<12} {'p-value':<12} {'Result'}")
    print("-" * 56)
    
    for sigma in noise_levels:
        if sigma not in all_surrogate_results[sig_type]:
            continue
        
        emp_list = all_surrogate_results[sig_type][sigma]
        
        X = np.array([[s['mean'], s['variance'], s['dominant_freq']] 
                       for s in emp_list])
        
        n = X.shape[0]
        p = X.shape[1]
        
        X_bar = np.mean(X, axis=0)
        S = np.cov(X.T)
        
        # Check if covariance is invertible
        if np.linalg.det(S) < 1e-20:
            print(f"{sigma:<8} {'—':<12} {'—':<12} {'—':<12} Covariance singular")
            continue
        
        diff = X_bar - mu_0
        T2 = n * diff @ np.linalg.inv(S) @ diff
        
        F_stat = (n - p) / (p * (n - 1)) * T2
        p_value = 1 - f_dist.cdf(F_stat, p, n - p)
        
        result = "MATCHES theory" if p_value > 0.05 else "DIFFERS from theory"
        print(f"{sigma:<8} {T2:<12.3f} {F_stat:<12.3f} {p_value:<12.4f} {result}")

print("\n'MATCHES' = failed to reject H0 (not proof of equivalence)")
print("'DIFFERS' = rejected H0 at 95% confidence")
print("NOTE: Exploratory analysis — 10 rollouts from 1 model per condition")

HOTELLING T² TEST (Exploratory)
Statistics: mean, variance, dominant frequency

--- SINUSOID ---
Reference: mean=0.0000, var=0.5000, freq=1.0000
σ        T²           F-stat       p-value      Result
--------------------------------------------------------
0.0      —            —            —            Covariance singular
0.1      —            —            —            Covariance singular
0.2      —            —            —            Covariance singular
0.5      —            —            —            Covariance singular
1.0      —            —            —            Covariance singular

--- TRIANGLE ---
Reference: mean=-0.0000, var=0.3336, freq=1.0000
σ        T²           F-stat       p-value      Result
--------------------------------------------------------
0.0      —            —            —            Covariance singular
0.1      9.043        2.345        0.1593       MATCHES theory
0.2      38.194       9.902        0.0065       DIFFERS from theory
0.5      —            —  